In [1]:
import torch
from torch import nn
from d2l import torch as d2l

In [13]:
batch_size = 256

train_iter,test_iter = d2l.load_data_fashion_mnist(batch_size)

num_inputs=784
num_outputs=10

W=torch.normal(0,0.01,size=(num_inputs,num_outputs),requires_grad=True)
B=torch.zeros(num_outputs,requires_grad=True)

def crossentropy(y_hat,y):
    return -torch.log(y_hat[range(len(y_hat)),y]).mean()

def softmax(X):
    X_exp=torch.exp(X)
    X_sum=X_exp.sum(dim=1,keepdim=True)
    return X_exp/X_sum

def net(X):
    return softmax(torch.matmul(X.reshape(-1,W.shape[0]),W)+B)

def accuracy(y_hat,y):
    y_hat=y_hat.argmax(axis=1)
    ac=y_hat.reshape(y.shape)==y
    return float(ac.type(y.dtype).sum())

trainer = torch.optim.SGD([W,B],lr=0.1)

epochs = 10

def train_ch3(epochs,net,loss,updater,train_iter,test_iter):
    for epoch in range(epochs):
        l,a = train_epoch_ch3(net,train_iter,loss,updater)
        print(f"epoch:{epoch} \t loss:{l:.3f} \t acc:{a:.3f}\n")

def train_epoch_ch3(net,train_iter,loss,updater):
    if isinstance(net,torch.nn.Module):
        net.train()
    metric=Accumulator(3)
    for X,y in train_iter:
        y_hat = net(X)
        l = loss(y_hat,y)
        if isinstance(updater,torch.optim.Optimizer):
            updater.zero_grad()
            l.backward()
            updater.step()
            metric.add(float(l)*len(y),accuracy(y_hat,y),y.numel())
        else:
            l.sum().backward()
            updater(X.shape[0])
            metric.add(float(l.sum()),accuracy(y_hat,y),y.numel())

    return metric[0]/metric[2],metric[1]/metric[2]

class Accumulator:
    def __init__(self,n):
        self.data = [0.0]*n

    def add(self,*args):
        self.data = [a+float(b) for a,b in zip(self.data,args)]

    def __getitem__(self, idx):
        return self.data[idx]

    def reset(self):
        self.data = [0.0]*len(self.data)


In [14]:
train_ch3(epochs,net,crossentropy,trainer,train_iter,test_iter)

epoch:0 	 loss:0.788 	 acc:0.748

epoch:1 	 loss:0.570 	 acc:0.813

epoch:2 	 loss:0.525 	 acc:0.826

epoch:3 	 loss:0.502 	 acc:0.832

epoch:4 	 loss:0.486 	 acc:0.837

epoch:5 	 loss:0.474 	 acc:0.840

epoch:6 	 loss:0.465 	 acc:0.844

epoch:7 	 loss:0.457 	 acc:0.846

epoch:8 	 loss:0.452 	 acc:0.847

epoch:9 	 loss:0.447 	 acc:0.849

